# Nautilus AquaVision v0 — training notebook

Outputs were stripped before publication so the repository does not redistribute source-dataset imagery. Set the `AQUAVISION_DATASET` environment variable to the merged YOLO dataset root when running outside the original Kaggle environment.

## Resources
- Kaggle notebook: https://www.kaggle.com/code/theushen/aquavision-v0-1-0-alpha
- Official dataset: https://doi.org/10.34740/kaggle/dsv/17844306


In [ ]:
%pip install -q -U ultralytics

import os
import csv
import json
import math
import random
import shutil
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import ultralytics

print("Python:", torch.__version__)
print("Ultralytics:", ultralytics.__version__)
print("CUDA:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("Ative GPU em Settings → Accelerator → GPU")

print(torch.cuda.get_device_name(0))
print(
    "VRAM:",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3,2),
    "GB"
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

INPUT_ROOT = Path(os.environ.get("AQUAVISION_DATASET", "/kaggle/input/datasets/theushen/nautilus-aquavision-v0"))

WORK = Path("/kaggle/working")
DATASET = WORK / "aquavision"

RUNS = WORK / "runs"

BASE_MODEL = "yolo26n.pt"

SMOKE_EPOCHS = 3
FULL_EPOCHS = 100

SMOKE_IMGSZ = 640
TRAIN_IMGSZ = 960

In [ ]:
if DATASET.exists():
    shutil.rmtree(DATASET)

DATASET.mkdir()

os.symlink(
    INPUT_ROOT / "images",
    DATASET / "images",
    target_is_directory=True
)

shutil.copytree(
    INPUT_ROOT / "labels",
    DATASET / "labels"
)

for f in [
    "data.yaml",
    "manifest.csv",
    "REPORT.md"
]:
    shutil.copy2(INPUT_ROOT / f, DATASET / f)

print(DATASET)

In [ ]:
IMG_EXT = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}

stats = {}

for split in [
    "train",
    "val",
    "test"
]:

    imgs = [
        p
        for p in (DATASET/"images"/split).iterdir()
        if p.suffix.lower() in IMG_EXT
    ]

    labels = list(
        (DATASET/"labels"/split).glob("*.txt")
    )

    print(split)
    print("Imagens:",len(imgs))
    print("Labels :",len(labels))

    assert len(imgs)==len(labels)

print("Dataset OK")

In [ ]:
YAML = WORK/"aquavision.yaml"

YAML.write_text(f"""
path: {DATASET}

train: images/train
val: images/val
test: images/test

names:
  0: floating_object
""")

print(YAML.read_text())

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image

imgs = list(
    (DATASET/"images/train").glob("*")
)

random.shuffle(imgs)

fig,axs=plt.subplots(3,4,figsize=(20,15))

for ax,img_path in zip(axs.flat,imgs[:12]):

    img=Image.open(img_path).convert("RGB")

    W,H=img.size

    label=DATASET/"labels/train"/f"{img_path.stem}.txt"

    ax.imshow(img)

    for line in label.read_text().splitlines():

        _,xc,yc,bw,bh=map(float,line.split())

        x=(xc-bw/2)*W
        y=(yc-bh/2)*H

        ax.add_patch(
            Rectangle(
                (x,y),
                bw*W,
                bh*H,
                fill=False
            )
        )

    ax.axis("off")

plt.show()

In [ ]:
from ultralytics import YOLO

model = YOLO(BASE_MODEL)

model.train(

    data=str(YAML),

    epochs=SMOKE_EPOCHS,

    imgsz=SMOKE_IMGSZ,

    batch=-1,

    device=0,

    project=str(RUNS),

    name="Smoke",

    plots=True
)

In [ ]:
model=YOLO(BASE_MODEL)

model.train(

    data=str(YAML),

    epochs=FULL_EPOCHS,

    imgsz=TRAIN_IMGSZ,

    batch=-1,

    device=0,

    patience=20,

    close_mosaic=10,

    optimizer="auto",

    cos_lr=True,

    project=str(RUNS),

    name="AquaVision",

    plots=True,

    save=True,

    save_period=10
)

In [ ]:
best = YOLO(
    RUNS/"AquaVision/weights/best.pt"
)

metrics = best.val(

    data=str(YAML),

    split="test",

    imgsz=TRAIN_IMGSZ
)

print(metrics.results_dict)

In [ ]:
best.predict(

    source=str(DATASET/"images/test"),

    imgsz=TRAIN_IMGSZ,

    conf=0.2,

    save=True
)

best.export(
    format="onnx"
)